# spatioloji_s — Basic Workflow Tutorial (10x Genomics Xenium)

This notebook covers the complete single-cell processing pipeline for **10x Genomics Xenium** spatial transcriptomics data using **spatioloji_s**.

## Key Xenium differences vs. CosMx / MERFISH

| Feature | Xenium | CosMx |
|---|---|---|
| Loading method | **`from_xenium(dir)`** — one call | `from_files()` with 5 separate files |
| Coordinate unit | **microns (µm)** | pixels |
| Coordinate system | **Single global field — no FOVs** | Per-FOV local + global |
| Expression format | **10x MEX / zarr.zip / tar.gz** (auto-detected) | CSV or NPZ |
| Cell ID column | **`cell_id`** | `cell` |
| Cell area column | **`cell_area`** (µm²) | `Area` (px²) |
| Image | **Single `morphology.ome.tif`** (OME-TIFF pyramid) | Per-FOV JPEGs |
| Negative controls | `NegControlProbe`, `NegControlCodeword`, `DeprecatedCodeword`, `UnassignedCodeword`, `Intergenic` | `NegProbe*` |

## Pipeline overview

```
Load Xenium directory  (sj.spatioloji.from_xenium)
  └─ Quick summary
  └─ Quality control (cells + genes)
     └─ Normalization (library size → log1p → scale)
        └─ Highly variable gene selection
           └─ Dimensionality reduction (PCA → UMAP)
              └─ Clustering (Leiden)
                 └─ Visualization (UMAP + spatial + image overlay)
                    └─ Save
```

## 0. Installation

```bash
pip install spatioloji-s
pip install "spatioloji-s[clustering]"   # Leiden (leidenalg + igraph)
pip install "spatioloji-s[reduction]"    # UMAP
pip install tifffile zarr imagecodecs    # OME-TIFF image support
```

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import spatioloji_s as sj

print(f"spatioloji_s version: {sj.__version__}")

---
## 1. Xenium Output Structure

A standard Xenium Onboard Analysis (XOA) output directory contains:

```
output-XeniumRun_<id>/
├── experiment.xenium              # JSON: run metadata, panel info, pixel_size
├── analysis_summary.html          # QC summary report
│
├── cells.csv.gz                   # Per-cell metadata  ← used by from_xenium
├── cell_boundaries.csv.gz         # Cell polygon vertices (µm)  ← used by from_xenium
├── nucleus_boundaries.csv.gz      # Nucleus polygon vertices (µm)
├── transcripts.csv.gz             # Per-transcript table (large; optional)
│
├── cell_feature_matrix/           # 10x MEX sparse format  ← used by from_xenium
│   ├── matrix.mtx.gz              #   genes × cells sparse count matrix
│   ├── barcodes.tsv.gz            #   cell barcodes (= cell_id)
│   └── features.tsv.gz            #   gene_id, gene_name, feature_type
├── cell_feature_matrix.zarr.zip   # Zarr format (alternative)  ← auto-detected
├── cell_feature_matrix.tar.gz     # Tar format (alternative)   ← auto-detected
│
├── morphology.ome.tif             # Whole-tissue DAPI Z-stack (OME-TIFF pyramid)
└── morphology_focus/              # 2D autofocus projections
    ├── morphology_focus_0000.ome.tif
    └── ...
```

### `cells.csv.gz` columns

| Column | Type | Description |
|---|---|---|
| `cell_id` | str | Unique cell identifier (**note: not `cell`**) |
| `x_centroid` | float | X centroid in **µm** |
| `y_centroid` | float | Y centroid in **µm** |
| `transcript_counts` | int | Number of detected gene transcripts |
| `control_probe_counts` | int | NegControlProbe transcript count |
| `control_codeword_counts` | int | NegControlCodeword count |
| `total_counts` | int | All transcripts including controls |
| `cell_area` | float | Cell area in **µm²** |
| `nucleus_area` | float | Nucleus area in **µm²** |

### `cell_boundaries.csv.gz` columns

| Column | Type | Description |
|---|---|---|
| `cell_id` | str | Cell identifier |
| `vertex_x` | float | Vertex X in **µm** |
| `vertex_y` | float | Vertex Y in **µm** |

> **No FOV system:** Xenium captures the entire tissue in a single continuous
> coordinate space. All coordinates are in microns (µm). There are no separate
> FOV tiles — `from_xenium` assigns `fov = "xenium"` as a single dummy FOV.

---
## 2. Load Data

### Option A — Load directly from Xenium output directory (first time)

`from_xenium()` auto-detects the expression format (MTX folder → zarr.zip → tar.gz),
loads cell boundaries as polygons, and stores the OME-TIFF path for later image access.

In [ ]:
# Path to your Xenium output directory
xenium_dir = "../../Xenium_5k/"

sp = sj.spatioloji.from_xenium(
    xenium_dir      = xenium_dir,
    load_boundaries = True,     # load cell_boundaries.csv.gz as polygon data
    matrix_type     = "tar",   # auto-detect: MTX folder → zarr.zip → tar.gz
    pixel_size      = 0.2125,   # µm per pixel for the full-res OME-TIFF (instrument default)
    lazy_load_image = True,     # store OME-TIFF path only; load pixels on demand
)
print(sp)

### Option B — Load a saved object (recommended after first load)

In [ ]:
# sp = sj.spatioloji.from_pickle("my_data/raw_xenium_spatioloji.pkl")
# print(sp)

---
## 3. Quick Summary

In [ ]:
sj.data.utils.quick_summary(sp)

In [ ]:
print("Cell meta columns:", sp.cell_meta.columns.tolist())
print(f"\nCells  : {sp.n_cells:,}")
print(f"Genes  : {sp.n_genes:,}")
print(f"FOVs   : {sp.n_fovs}  (always 1 for Xenium — single global coordinate space)")

# Xenium-specific attributes stored by from_xenium
print(f"\nXenium directory  : {sp._xenium_dir}")
print(f"OME-TIFF path     : {sp._xenium_image_path}")
print(f"Pixel size (µm/px): {sp._xenium_pixel_size}")

In [ ]:
# Identify negative-control genes in the expression matrix.
# Xenium uses five prefixes for non-biological features:
#   NegControlProbe     — off-target probe binding (background signal)
#   NegControlCodeword  — erroneous barcode decoding
#   DeprecatedCodeword  — retired codewords from older panel versions
#   UnassignedCodeword  — valid codewords not assigned to any gene
#   Intergenic          — transcripts decoded outside annotated gene bodies
NEG_PREFIXES = (
    "NegControlProbe",
    "NegControlCodeword",
    "DeprecatedCodeword",
    "UnassignedCodeword",
    "Intergenic",
)

neg_genes  = [g for g in sp.gene_index if g.startswith(NEG_PREFIXES)]
real_genes = [g for g in sp.gene_index if not g.startswith(NEG_PREFIXES)]

print(f"Total features      : {sp.n_genes:,}")
print(f"Panel genes         : {len(real_genes):,}")
print(f"Negative controls   : {len(neg_genes):,}")
print(f"\nControl gene names  : {neg_genes}")

---
## 4. Quality Control

### Xenium QC differences vs. CosMx

| QC metric | Xenium | CosMx |
|---|---|---|
| Cell area column | **`cell_area`** (µm²) | `Area` (px²) |
| Negative control prefixes | `NegControlProbe`, `NegControlCodeword`, `DeprecatedCodeword`, `UnassignedCodeword`, `Intergenic` | `NegProbe` |
| FOV-level QC | **Not applicable** (single coordinate space) | `qc_fov_metrics` |
| Typical counts/cell | 50–500 (sparse panel) | 100–2,000 |

### Xenium negative-control gene types

| Prefix | Biological meaning | Use in QC |
|---|---|---|
| `NegControlProbe` | Off-target probe binding | Primary background measure |
| `NegControlCodeword` | Erroneous barcode decoding | Decoding error rate |
| `DeprecatedCodeword` | Retired codewords (old panel versions) | Carry-over artefact |
| `UnassignedCodeword` | Valid codewords with no gene assignment | Panel coverage metric |
| `Intergenic` | Transcripts outside annotated gene bodies | Annotation / segmentation artefact |

> **Tip:** Xenium panels are typically smaller (300–5,000 genes) with lower per-cell
> counts than CosMx. Adjust `total_counts_min` accordingly (10–50 is typical).

In [ ]:
NEG_PREFIXES = (
    "NegControlProbe",
    "NegControlCodeword",
    "DeprecatedCodeword",
    "UnassignedCodeword",
    "Intergenic",
)

qc_config = sj.XeniumQCConfig(
    neg_prefixes              = NEG_PREFIXES,
    # Cell filters
    transcript_counts_min     = 10,     # min panel-gene transcripts per cell
    n_features_min            = 5,      # min unique genes detected
    pct_counts_neg_max        = 0.05,   # max 5% from any control type (XOA: warn 4%, error 10%)
    pct_counts_mt_max         = 0.25,   # max 25% mitochondrial
    cell_area_min             = 20.0,   # µm²  — remove debris
    cell_area_max             = 500.0,  # µm²  — remove putative doublets
    nucleus_cell_ratio_max    = 1.0,    # remove segmentation errors
    # Gene filters
    gene_filter_method        = "percentile",
    gene_percentile_threshold = 50,     # keep genes above 50th pct of ctrl baseline
    output_dir                = "./xenium_qc_output/",
    save_plots                = True,
)

In [ ]:
qc = sj.xenium_qc(sp, config=qc_config)

In [ ]:
# xenium_qc.__init__ automatically computes QC columns and adds them to sp.cell_meta.
# Preview all metrics that were added:
qc_cols = [
    c for c in sp.cell_meta.columns
    if c.startswith((
        "pct_counts_",
        "n_features",
        "total_counts_expr",
        "transcript_counts_expr",
        "nucleus_cell_ratio",
        "transcript_density",
        "QC_",
        "is_neg_ctrl",
        "neg_ctrl_type",
    ))
]
print(f"QC metrics added to cell_meta ({len(qc_cols)} columns):")
print(qc_cols)
sp.cell_meta[qc_cols].describe().round(4)

In [ ]:
# --- Option A: run all steps at once ---
# sp_filtered = qc.run_qc_pipeline(plot=True)

# --- Option B: run steps individually for more control ---

# Cell & nucleus area (debris < 20 µm², doublets > 500 µm², ratio > 1 = seg. error)
qc.qc_cell_area(plot=True)

# Transcript count + feature diversity distributions
qc.qc_transcript_metrics(plot=True)

# Per-type control fraction breakdown (NegControlProbe, NegControlCodeword, etc.)
qc.qc_control_metrics(plot=True)

# Nucleus : cell area ratio — segmentation quality
qc.qc_nucleus_cell_ratio(plot=True)

# Transcript density — spatial uniformity (transcripts / µm²)
qc.qc_transcript_density(plot=True)

In [ ]:
# filter_cells() applies all thresholds from XeniumQCConfig and prints per-filter counts.
# Pass custom_filters for additional per-column thresholds.
cell_mask = qc.filter_cells(
    custom_filters={
        # Add extra per-column filters if needed, e.g.:
        # "transcript_density": (0.01, None),  # min 0.01 tx/µm²
    }
)

# filter_genes() excludes all control genes, then applies panel-gene filter
gene_mask = qc.filter_genes(plot=True)

print(f"\nCells passing QC: {cell_mask.sum():,} / {len(cell_mask):,}")
print(f"Genes passing QC: {gene_mask.sum():,} / {len(gene_mask):,}")

In [ ]:
# Print a tabular QC summary and save to JSON + bar chart
qc.summarize_qc(plot=True)

In [ ]:
sp_filtered = qc.apply_filters()

print(f"Before QC: {sp.n_cells:,} cells, {sp.n_genes:,} genes")
print(f"After QC : {sp_filtered.n_cells:,} cells, {sp_filtered.n_genes:,} genes")

In [ ]:
os.makedirs("my_data", exist_ok=True)
sp_filtered.to_pickle("my_data/filtered_xenium_spatioloji.pkl")

sp = sp_filtered

---
## 5. Normalization

Same three-step workflow as CosMx. Library-size normalisation to 10,000 (CPM-equivalent)
followed by log1p and z-score scaling for PCA.

In [ ]:
# Step 1 — library-size normalisation
sj.processing.normalize_total(sp, target_sum=1e4, inplace=True)
# Adds layer: 'normalized_counts'

# Step 2 — log1p transform
sj.processing.log_transform(sp, layer="normalized_counts", inplace=True)
# Adds layer: 'log_normalized'

# Step 3 — z-score scaling (PCA input only; do NOT use for DE / violin plots)
sj.processing.scale(sp, layer="log_normalized", max_value=10.0, inplace=True)
# Adds layer: 'scaled'

print("Available layers:", sp.list_layers())

---
## 6. Highly Variable Gene (HVG) Selection

For Xenium panels (typically 300–5,000 genes) with sparse counts, the **deviance** method
is recommended. If the panel is small (<500 genes), skip HVG selection and use all genes.

In [ ]:
n_panel_genes = sp.n_genes
n_top_genes   = min(2000, n_panel_genes)

if n_panel_genes <= 500:
    # Small panel — use all genes for PCA
    sp.gene_meta["highly_variable"] = True
    print(f"Panel ≤ 500 genes — using all {n_panel_genes} genes")
else:
    # Deviance: designed for sparse count data, best for Xenium panels
    sj.processing.highly_variable_genes(
        sp,
        layer       = "normalized_counts",
        method      = "deviance",
        n_top_genes = n_top_genes,
        inplace     = True,
    )
    n_hvg = sp.gene_meta["highly_variable"].sum()
    print(f"HVGs selected: {n_hvg} / {n_panel_genes}")

In [ ]:
# Optional: compare multiple HVG methods to check consistency
if n_panel_genes > 500:
    hvg_comparison = sj.processing.compare_hvg_methods(
        sp,
        methods     = ["seurat_v3", "deviance", "pearson_residuals"],
        n_top_genes = n_top_genes,
        layer       = "normalized_counts",
    )
    consensus_genes = hvg_comparison.index[hvg_comparison["consensus"]].tolist()
    print(f"Consensus HVGs (majority vote): {len(consensus_genes)}")
    hvg_comparison.sort_values("n_methods_selected", ascending=False).head(20)

---
## 7. Dimensionality Reduction

### 7a. PCA

In [ ]:
sj.processing.pca(
    sp,
    layer               = "scaled",
    use_highly_variable = True,
    n_comps             = 50,
    random_state        = 42,
    inplace             = True,
)

sj.processing.plot_pca_variance(sp, n_pcs=30)

### 7b. UMAP

In [ ]:
# Requires: pip install "spatioloji-s[reduction]"
sj.processing.umap(
    sp,
    use_pca      = True,
    n_pcs        = 20,
    n_neighbors  = 30,
    min_dist     = 0.3,
    random_state = 42,
    inplace      = True,
)
# Stores UMAP1 / UMAP2 in sp.cell_meta and X_umap in sp._embeddings

---
## 8. Clustering

### 8a. Leiden resolution sweep

In [ ]:
sweep_df = sj.processing.leiden_resolution_sweep(
    sp,
    resolutions  = list(np.arange(0.2, 1.6, 0.2)),
    n_runs       = 5,
    n_pcs        = 20,
    n_neighbors  = 15,
    random_state = 42,
)
sweep_df

In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 4))
ax2 = ax1.twinx()

ax1.plot(sweep_df["resolution"], sweep_df["mean_ari"], "o-", color="steelblue", label="Mean ARI (stability)")
ax1.fill_between(
    sweep_df["resolution"],
    sweep_df["mean_ari"] - sweep_df["std_ari"],
    sweep_df["mean_ari"] + sweep_df["std_ari"],
    alpha=0.2, color="steelblue",
)
ax2.plot(sweep_df["resolution"], sweep_df["n_clusters_mean"], "s--", color="coral", label="N clusters")

ax1.set_xlabel("Resolution")
ax1.set_ylabel("Mean ARI (stability)", color="steelblue")
ax2.set_ylabel("N clusters", color="coral")
ax1.set_title("Leiden Resolution Sweep — Xenium")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="lower left")
plt.tight_layout()
plt.savefig("my_data/leiden_resolution_sweep.pdf", bbox_inches="tight")
plt.show()

In [ ]:
optimal_res = float(sweep_df.loc[sweep_df["mean_ari"].idxmax(), "resolution"])
print(f"Most stable resolution: {optimal_res}")
# Override if you prefer a specific cluster count:
# optimal_res = 0.5

### 8b. Run Leiden clustering

In [ ]:
sj.processing.leiden_clustering(
    sp,
    layer               = "scaled",
    use_highly_variable = True,
    resolution          = optimal_res,
    n_neighbors         = 15,
    n_pcs               = 50,
    use_pca             = True,
    random_state        = 42,
    output_column       = "leiden",
    inplace             = True,
)

print(f"Leiden clusters: {sp.cell_meta['leiden'].nunique()}")
print(sp.cell_meta["leiden"].value_counts().sort_index())

### 8c. Assess clustering quality

In [ ]:
metrics = sj.processing.assess_clustering_quality(
    sp,
    cluster_col = "leiden",
    layer       = "scaled",
    n_pcs       = 20,
    sample_size = 5000,
)

for k, v in metrics.items():
    if k != "cluster_sizes":
        print(f"  {k}: {v}")

---
## 9. UMAP Visualization

### 9a. UMAP coloured by cluster

In [ ]:
os.makedirs("my_plots", exist_ok=True)

sj.visualization.plot_umap(
    sp,
    color_by   = "leiden",
    palette    = "tab20",
    point_size = 3.0,
    title      = "UMAP — Leiden clusters (Xenium)",
    show       = True,
    save_path  = "my_plots/UMAP_leiden.pdf",
)

In [ ]:
# transcript_counts: gene transcripts only (excludes negative controls)
sj.visualization.plot_umap(
    sp,
    color_by  = "transcript_counts",
    color_map = "viridis",
    title     = "UMAP — Transcript counts",
    show      = True,
    save_path = "my_plots/UMAP_transcript_counts.pdf",
)

In [ ]:
# Gene expression overlay — replace 'EPCAM' with a gene from your panel
sj.visualization.plot_umap(
    sp,
    gene      = "EPCAM",
    layer     = "log_normalized",
    color_map = "Reds",
    title     = "UMAP — EPCAM expression",
    show      = True,
    save_path = "my_plots/UMAP_EPCAM.pdf",
)

### 9b. Multi-panel UMAP grid

In [ ]:
# Adjust gene list to markers present in your Xenium panel.
# Also overlay QC-derived columns to spot technical outliers per cluster.
sj.visualization.plot_umap_grid(
    sp,
    genes    = ["EPCAM", "CD3D", "CD14", "MS4A1"],
    features = [
        "transcript_counts",       # library size (raw)
        "n_features_by_counts",    # gene diversity
        "pct_counts_neg",          # aggregate negative-control fraction
        "nucleus_cell_ratio",      # segmentation quality
        "transcript_density",      # spatial expression density (tx/µm²)
        "cell_area",               # cell size
    ],
    ncols    = 3,
    layer    = "log_normalized",
    show     = True,
    save_path = "my_plots/UMAP_grid.pdf",
)

### 9c. Violin, heatmap, dot plots

In [ ]:
marker_genes = ["EPCAM", "CD3D", "CD14", "MS4A1", "CD68", "DCN"]
marker_genes = [g for g in marker_genes if g in sp.gene_index.tolist()]

sj.visualization.plot_violin(
    sp,
    genes    = marker_genes,
    group_by = "leiden",
    layer    = "log_normalized",
    show     = True,
    save_path = "my_plots/violin_markers.pdf",
)

In [ ]:
gene_order = [
    "EPCAM", "KRT18", "KRT19",          # Epithelial
    "CD3D", "CD3E", "CD8A", "CD4",      # T cells
    "MS4A1", "CD79A",                    # B cells
    "CD14", "CD68", "LYZ",              # Myeloid
    "DCN", "COL1A1",                     # Fibroblast
]
gene_order    = [g for g in gene_order if g in sp.gene_index.tolist()]
cluster_order = [str(i) for i in sorted(sp.cell_meta["leiden"].unique())]

sj.visualization.plot_heatmap(
    sp,
    genes       = gene_order,
    group_by    = "leiden",
    gene_order  = gene_order,
    group_order = cluster_order,
    scale       = "row",
    layer       = "log_normalized",
    show        = True,
    save_path   = "my_plots/heatmap_markers.pdf",
)

sj.visualization.plot_dotplot(
    sp,
    genes      = gene_order,
    gene_order = gene_order,
    group_by   = "leiden",
    layer      = "log_normalized",
    show       = True,
    save_path  = "my_plots/dotplot_markers.pdf",
)

---
## 10. Spatial Visualization

Xenium data lives in a **single continuous coordinate space** (µm), so all spatial
plots use global coordinates directly. The OME-TIFF pyramid stored in
`sp._xenium_image_path` can be opened at any resolution level via
`sp.get_xenium_image(level=...)` — level `"0"` is full resolution,
higher levels are progressively downsampled (factor 2× per level).

### 10a. Spatial scatter — cluster map

In [ ]:
cluster_ids     = sp.cell_meta["leiden"].astype(str)
unique_clusters = sorted(cluster_ids.unique())
cmap            = plt.get_cmap("tab20", len(unique_clusters))

fig, ax = plt.subplots(figsize=(10, 8))
for i, cl in enumerate(unique_clusters):
    mask = cluster_ids == cl
    ax.scatter(
        sp.cell_meta.loc[mask, "x_centroid"],
        sp.cell_meta.loc[mask, "y_centroid"],
        s=1, c=[cmap(i)], label=cl, alpha=0.6, linewidths=0,
    )

ax.set_aspect("equal")
ax.invert_yaxis()          # OME-TIFF origin is top-left; match image convention
ax.set_xlabel("X (µm)")
ax.set_ylabel("Y (µm)")
ax.set_title("Spatial map — Leiden clusters (Xenium)")
ax.legend(markerscale=6, loc="upper right", ncol=2, fontsize=7)
plt.tight_layout()
plt.savefig("my_plots/spatial_leiden.pdf", bbox_inches="tight")
plt.show()

### 10b. Spatial gene expression map

In [ ]:
gene = "EPCAM"   # replace with a gene in your panel

if gene in sp.gene_index:
    gene_expr = sp.get_expression(gene, layer="log_normalized")

    fig, ax = plt.subplots(figsize=(10, 8))
    sc = ax.scatter(
        sp.cell_meta["x_centroid"],
        sp.cell_meta["y_centroid"],
        c=gene_expr, cmap="Reds", s=1, alpha=0.8, linewidths=0,
    )
    plt.colorbar(sc, ax=ax, label="log-normalised expression")
    ax.set_aspect("equal")
    ax.invert_yaxis()
    ax.set_xlabel("X (µm)")
    ax.set_ylabel("Y (µm)")
    ax.set_title(f"Spatial expression — {gene}")
    plt.tight_layout()
    plt.savefig(f"my_plots/spatial_{gene}.pdf", bbox_inches="tight")
    plt.show()
else:
    print(f"{gene} not in panel — replace with a gene from sp.gene_index")

### 10c. Cell overlay on the OME-TIFF morphology image

`sp.get_xenium_image(level)` opens the `morphology.ome.tif` OME-TIFF pyramid via zarr
and returns an array of shape **(n_channels, H, W)**.

| Level | Effective pixel size | Typical image size |
|---|---|---|
| `"0"` | 0.2125 µm/px (full res) | ~50,000 × 50,000 per channel |
| `"2"` | ~0.85 µm/px | ~12,500 × 12,500 |
| `"3"` | ~1.70 µm/px | ~6,250 × 6,250 |
| `"4"` | ~3.40 µm/px | ~3,125 × 3,125 |

> Use level `"3"` or `"4"` for whole-tissue overview plots.  
> Use level `"0"` or `"1"` only for small ROI crops (full array is very large).

In [ ]:
# Requires: pip install tifffile zarr imagecodecs
# Load a downsampled level for a whole-tissue overview
LEVEL = "3"

img = sp.get_xenium_image(level=LEVEL)
print(f"Image shape (channels, H, W): {img.shape}")

# Effective pixel size at this level
px_size = sp._xenium_pixel_size * (2 ** int(LEVEL))  # µm per pixel
print(f"Effective pixel size: {px_size:.4f} µm/px")

In [ ]:
# DAPI channel is index 0; adjust if your panel has a different channel order
dapi = img[0]  # shape (H, W)

# Convert cell centroids from µm to pixels at this level
x_px = sp.cell_meta["x_centroid"].values / px_size
y_px = sp.cell_meta["y_centroid"].values / px_size

# Colour cells by Leiden cluster
cluster_ids     = sp.cell_meta["leiden"].astype(str)
unique_clusters = sorted(cluster_ids.unique())
cmap            = plt.get_cmap("tab20", len(unique_clusters))
color_map       = {cl: cmap(i) for i, cl in enumerate(unique_clusters)}
colors          = [color_map[cl] for cl in cluster_ids]

fig, ax = plt.subplots(figsize=(12, 10))

# Display DAPI background (percentile clipping to avoid bright artefacts)
vmin, vmax = np.percentile(dapi, [1, 99])
ax.imshow(dapi, cmap="gray", vmin=vmin, vmax=vmax, origin="upper")

# Overlay cell centroids
ax.scatter(x_px, y_px, c=colors, s=1, alpha=0.5, linewidths=0)

ax.set_xlabel(f"X (pixels at level {LEVEL}, {px_size:.2f} µm/px)")
ax.set_ylabel(f"Y (pixels at level {LEVEL}, {px_size:.2f} µm/px)")
ax.set_title(f"Leiden clusters overlaid on DAPI (level {LEVEL})")
ax.axis("off")
plt.tight_layout()
plt.savefig("my_plots/spatial_leiden_on_dapi.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# --- ROI crop example ---
# Select a region in µm and convert to pixel coordinates at the chosen level

# Define ROI in µm (adjust to a region of interest in your data)
roi_x_um = (1000, 2000)   # µm
roi_y_um = (1000, 2000)   # µm

# Convert to pixel coordinates at this level
roi_x_px = (int(roi_x_um[0] / px_size), int(roi_x_um[1] / px_size))
roi_y_px = (int(roi_y_um[0] / px_size), int(roi_y_um[1] / px_size))

dapi_crop = dapi[roi_y_px[0]:roi_y_px[1], roi_x_px[0]:roi_x_px[1]]

# Filter cells in this ROI
roi_mask = (
    (sp.cell_meta["x_centroid"] >= roi_x_um[0]) &
    (sp.cell_meta["x_centroid"] <  roi_x_um[1]) &
    (sp.cell_meta["y_centroid"] >= roi_y_um[0]) &
    (sp.cell_meta["y_centroid"] <  roi_y_um[1])
)
roi_meta = sp.cell_meta[roi_mask]

# Convert ROI cell coordinates to cropped-image pixel coordinates
roi_x_px_cells = (roi_meta["x_centroid"].values - roi_x_um[0]) / px_size
roi_y_px_cells = (roi_meta["y_centroid"].values - roi_y_um[0]) / px_size
roi_colors     = [color_map[cl] for cl in roi_meta["leiden"].astype(str)]

fig, ax = plt.subplots(figsize=(8, 8))
vmin, vmax = np.percentile(dapi_crop, [1, 99])
ax.imshow(dapi_crop, cmap="gray", vmin=vmin, vmax=vmax, origin="upper")
ax.scatter(roi_x_px_cells, roi_y_px_cells, c=roi_colors, s=10, alpha=0.7, linewidths=0)
ax.set_title(f"ROI {roi_x_um}–{roi_y_um} µm — Leiden clusters on DAPI (level {LEVEL})")
ax.axis("off")
plt.tight_layout()
plt.savefig("my_plots/spatial_leiden_roi.png", dpi=200, bbox_inches="tight")
plt.show()

---
## 11. Save the Final Object

In [ ]:
sp.to_pickle("my_data/processed_xenium_spatioloji.pkl")
print("Saved.")

In [ ]:
print("=" * 55)
print("Processed Xenium object summary")
print("=" * 55)
print(f"  Cells           : {sp.n_cells:,}")
print(f"  Genes           : {sp.n_genes:,}")
print(f"  Layers          : {sp.list_layers()}")
print(f"  Leiden clusters : {sp.cell_meta['leiden'].nunique()}")
print(f"  OME-TIFF        : {sp._xenium_image_path}")
print("\nNext steps:")
print("  → spatial_analysis.ipynb — neighbourhood enrichment, Moran's I, Ripley K")
print("  → ccc.ipynb              — cell-cell communication (3-layer polygon framework)")

---
## Appendix: Xenium Reference

### `from_xenium()` parameter reference

| Parameter | Default | Description |
|---|---|---|
| `xenium_dir` | — | Path to Xenium output directory |
| `load_boundaries` | `True` | Load `cell_boundaries.csv.gz` as polygon data |
| `matrix_type` | `"auto"` | `"auto"` / `"mtx"` / `"zarr"` / `"tar"` |
| `pixel_size` | `0.2125` | µm per pixel for the full-res OME-TIFF |
| `lazy_load_image` | `True` | Store OME-TIFF path only; load on demand |

### `get_xenium_image(level)` reference

| Level | Pixel size (µm/px) | Recommended use |
|---|---|---|
| `"0"` | 0.2125 (full res) | Small ROI crops only |
| `"1"` | 0.425 | Medium ROI |
| `"2"` | 0.85 | Large ROI |
| `"3"` | 1.70 | **Whole-tissue overview** |
| `"4"` | 3.40 | Fast whole-tissue thumbnail |

Returns shape **(n_channels, H, W)**. Channel 0 is typically DAPI.

### `XeniumQCConfig` parameter reference

| Parameter | Default | Description |
|---|---|---|
| `neg_prefixes` | 5-tuple of all control prefixes | Control gene prefixes to exclude |
| `transcript_counts_min` | `10` | Min panel-gene transcripts per cell |
| `n_features_min` | `5` | Min unique genes detected per cell |
| `pct_counts_neg_max` | `0.05` | Max fraction of any control type (XOA: warn 4%, error 10%) |
| `pct_counts_mt_max` | `0.25` | Max mitochondrial fraction |
| `cell_area_min` | `20.0` | Min cell area (µm²) — remove debris |
| `cell_area_max` | `500.0` | Max cell area (µm²) — remove doublets |
| `nucleus_cell_ratio_max` | `1.0` | Max nucleus:cell area ratio — remove segmentation errors |
| `gene_filter_method` | `"percentile"` | `"percentile"` / `"absolute"` / `"min_cells"` |
| `gene_percentile_threshold` | `50` | Keep genes above this percentile of ctrl baseline |
| `gene_min_cells` | `None` | Min cells expressing gene (used when method=`"min_cells"`) |
| `output_dir` | `"./xenium_qc_output/"` | Directory for saved plots and JSON report |
| `save_plots` | `True` | Whether to auto-save figures to `output_dir` |

### `xenium_qc` method reference

| Method | Description |
|---|---|
| `qc_cell_area(plot)` | Cell area + nucleus area histograms, nucleus:cell scatter |
| `qc_transcript_metrics(plot)` | Transcript count + gene diversity distributions |
| `qc_control_metrics(plot)` | Per-type control fraction bar chart + violin (XOA thresholds) |
| `qc_nucleus_cell_ratio(plot)` | Ratio histogram + CDF; flags ratio > 1.0 |
| `qc_transcript_density(plot)` | Density histogram + spatial scatter coloured by tx/µm² |
| `filter_cells(custom_filters)` | Apply all thresholds; print per-filter failure counts |
| `filter_genes(plot)` | Exclude controls; filter panel genes by configured method |
| `summarize_qc(plot)` | Print table, save JSON report, plot cell-counts bar chart |
| `apply_filters(filter_cells, filter_genes)` | Return filtered `spatioloji`; propagates `_xenium_*` attrs |
| `run_qc_pipeline(plot)` | Run all 9 steps above in sequence |

### QC metrics added to `cell_meta` by `xenium_qc.__init__`

| Column | Description |
|---|---|
| `total_counts_expr` | Total panel-gene (non-control) counts |
| `transcript_counts_expr` | Same as `total_counts_expr` (alias) |
| `n_features_by_counts` | Number of unique genes detected |
| `pct_counts_neg` | Fraction of total counts from all control genes combined |
| `pct_counts_mt` | Fraction of total counts from mitochondrial genes |
| `pct_counts_NegControlProbe` | Fraction from `NegControlProbe` genes only |
| `pct_counts_NegControlCodeword` | Fraction from `NegControlCodeword` genes only |
| `pct_counts_DeprecatedCodeword` | Fraction from `DeprecatedCodeword` genes only |
| `pct_counts_UnassignedCodeword` | Fraction from `UnassignedCodeword` genes only |
| `pct_counts_Intergenic` | Fraction from `Intergenic` genes only |
| `nucleus_cell_ratio` | `nucleus_area / cell_area` — segmentation quality indicator |
| `transcript_density` | `transcript_counts / cell_area` (tx/µm²) — spatial expression density |

### Xenium-specific `cell_meta` columns (from `cells.csv.gz`)

| Column | Description |
|---|---|
| `x_centroid` | Cell centroid X in µm |
| `y_centroid` | Cell centroid Y in µm |
| `transcript_counts` | Gene transcripts (excludes controls) |
| `control_probe_counts` | `NegControlProbe` count |
| `control_codeword_counts` | `NegControlCodeword` count |
| `total_counts` | All transcripts including controls |
| `cell_area` | Cell area in µm² |
| `nucleus_area` | Nucleus area in µm² |
| `fov` | Always `"xenium"` (single global space) |

### Xenium negative-control gene prefixes

| Prefix | Meaning |
|---|---|
| `NegControlProbe` | Off-target probe binding — primary background signal |
| `NegControlCodeword` | Erroneous barcode decoding |
| `DeprecatedCodeword` | Retired codewords from older panel versions |
| `UnassignedCodeword` | Valid codewords with no gene assignment |
| `Intergenic` | Transcripts decoded outside annotated gene bodies |

### HVG method guide (Xenium)

| Method | Recommended for |
|---|---|
| `deviance` | **Best for sparse Xenium panels (< 1,000 genes)** |
| `seurat_v3` | Larger panels (> 1,000 genes) |
| `pearson_residuals` | Correcting for technical capture-rate variation |

### QC threshold guide (Xenium)

| Parameter | Typical value | Notes |
|---|---|---|
| `transcript_counts_min` | 10–50 | Lower than CosMx due to sparse counts |
| `n_features_min` | 5–20 | Minimum gene diversity |
| `pct_counts_neg_max` | 3–10% | Aggregate from all five control prefixes; XOA warns at 4%, errors at 10% |
| `cell_area_min` / `cell_area_max` | 20–500 µm² | Filter debris (< 20) and doublets (> 500) |
| `nucleus_cell_ratio_max` | 1.0 | Ratio > 1.0 = nucleus larger than cell = segmentation error |
| `transcript_density` | > 0.01 tx/µm² | Optional spatial uniformity filter via `custom_filters` |